as

In [3]:
import os

# Muestra la carpeta actual donde está tu notebook
print("Carpeta actual:", os.getcwd())

# Lista los archivos y carpetas que hay en la carpeta actual
print("\nArchivos y carpetas en la carpeta actual:")
print(os.listdir('.'))

Carpeta actual: d:\2026.1\IA\SIS420\PrimerParcial\PrimeraParte\Dataset2

Archivos y carpetas en la carpeta actual:
['ADMISSIONS.csv', 'CALLOUT.csv', 'CAREGIVERS.csv', 'CHARTEVENTS.csv', 'CPTEVENTS.csv', 'dataset2.ipynb', 'DATETIMEEVENTS.csv', 'DIAGNOSES_ICD.csv', 'DRGCODES.csv', 'D_CPT.csv', 'D_ICD_DIAGNOSES.csv', 'D_ICD_PROCEDURES.csv', 'D_ITEMS.csv', 'D_LABITEMS.csv', 'ICUSTAYS.csv', 'INPUTEVENTS_CV.csv', 'INPUTEVENTS_MV.csv', 'LABEVENTS.csv', 'LICENSE.txt', 'MICROBIOLOGYEVENTS.csv', 'NOTEEVENTS.csv', 'OUTPUTEVENTS.csv', 'PATIENTS.csv', 'PRESCRIPTIONS.csv', 'PROCEDUREEVENTS_MV.csv', 'PROCEDURES_ICD.csv', 'SERVICES.csv', 'SHA256SUMS.txt', 'TRANSFERS.csv']


In [5]:
import pandas as pd

# Ruta corregida con barra al final
path = r'd:\2026.1\IA\SIS420\PrimerParcial\PrimeraParte\Dataset2\\'

# Cargar las tablas principales
patients   = pd.read_csv(path + 'PATIENTS.csv')
admissions = pd.read_csv(path + 'ADMISSIONS.csv')
icustays   = pd.read_csv(path + 'ICUSTAYS.csv')
diagnoses  = pd.read_csv(path + 'DIAGNOSES_ICD.csv')
labevents  = pd.read_csv(path + 'LABEVENTS.csv')

# Mostrar información básica
print("=== PATIENTS ===")
print(patients.shape)
print(patients.columns.tolist())
print(patients.head(3))

print("\n=== ADMISSIONS ===")
print(admissions.shape)
print(admissions.columns.tolist())
print(admissions.head(3))

print("\n=== ICUSTAYS ===")
print(icustays.shape)
print(icustays.columns.tolist())
print(icustays.head(3))

=== PATIENTS ===
(100, 8)
['row_id', 'subject_id', 'gender', 'dob', 'dod', 'dod_hosp', 'dod_ssn', 'expire_flag']
   row_id  subject_id gender                  dob                  dod  \
0    9467       10006      F  2094-03-05 00:00:00  2165-08-12 00:00:00   
1    9472       10011      F  2090-06-05 00:00:00  2126-08-28 00:00:00   
2    9474       10013      F  2038-09-03 00:00:00  2125-10-07 00:00:00   

              dod_hosp              dod_ssn  expire_flag  
0  2165-08-12 00:00:00  2165-08-12 00:00:00            1  
1  2126-08-28 00:00:00                  NaN            1  
2  2125-10-07 00:00:00  2125-10-07 00:00:00            1  

=== ADMISSIONS ===
(129, 19)
['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospital_expire_flag', 'has_chartevents_data']
   row_id  subject_id  hadm_id  

In [6]:
import pandas as pd

# Ruta (la que ya te funcionó)
path = r'd:\2026.1\IA\SIS420\PrimerParcial\PrimeraParte\Dataset2\\'

# Cargar solo las tablas necesarias
patients   = pd.read_csv(path + 'PATIENTS.csv')
admissions = pd.read_csv(path + 'ADMISSIONS.csv')
icustays   = pd.read_csv(path + 'ICUSTAYS.csv')
diagnoses  = pd.read_csv(path + 'DIAGNOSES_ICD.csv')

print("Tablas cargadas correctamente")

# ============================
# 1. Unir PATIENTS + ADMISSIONS
# ============================
df = admissions.merge(patients[['subject_id', 'gender', 'dob', 'dod']], 
                       on='subject_id', how='left')

# Calcular edad aproximada en años
df['dob'] = pd.to_datetime(df['dob'])
df['admittime'] = pd.to_datetime(df['admittime'])
df['age'] = (df['admittime'] - df['dob']).dt.days // 365
df['age'] = df['age'].clip(upper=90)   # tope de 90 años (común en MIMIC)

# Crear variable objetivo: ¿falleció en el hospital?
df['mortality'] = df['hospital_expire_flag'].astype(int)

# ============================
# 2. Agregar información de ICUSTAYS
# ============================
icu_summary = icustays.groupby('hadm_id').agg({
    'los': 'sum',                    # duración total en UCI
    'first_careunit': 'first'        # tipo de UCI
}).reset_index()

df = df.merge(icu_summary, on='hadm_id', how='left')

# ============================
# 3. Agregar número de diagnósticos
# ============================
diag_count = diagnoses.groupby('hadm_id').size().reset_index(name='num_diagnoses')
df = df.merge(diag_count, on='hadm_id', how='left')

# ============================
# Mostrar resultado final
# ============================
print("\n=== DATASET FINAL ===")
print(df.shape)
print(df.columns.tolist())
print("\nPrimeras 5 filas:")
print(df.head())

# Guardar el dataset procesado
df.to_csv('mimic_mortality_ready.csv', index=False)
print("\nDataset guardado como 'mimic_mortality_ready.csv'")


Tablas cargadas correctamente

=== DATASET FINAL ===
(129, 27)
['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospital_expire_flag', 'has_chartevents_data', 'gender', 'dob', 'dod', 'age', 'mortality', 'los', 'first_careunit', 'num_diagnoses']

Primeras 5 filas:
   row_id  subject_id  hadm_id           admittime            dischtime  \
0   12258       10006   142345 2164-10-23 21:09:00  2164-11-01 17:15:00   
1   12263       10011   105331 2126-08-14 22:32:00  2126-08-28 18:59:00   
2   12265       10013   165520 2125-10-04 23:36:00  2125-10-07 15:13:00   
3   12269       10017   199207 2149-05-26 17:19:00  2149-06-03 18:42:00   
4   12270       10019   177759 2163-05-14 20:43:00  2163-05-15 12:00:00   

             deathtime admission_type         admission_location  \
0                  Na

In [7]:
import pandas as pd
import numpy as np

# Cargar el dataset que acabamos de crear
df = pd.read_csv('mimic_mortality_ready.csv')

print("Dimensiones originales:", df.shape)

# ====================== LIMPIEZA Y FEATURE ENGINEERING ======================

# 1. Convertir fechas a datetime
date_cols = ['admittime', 'dischtime', 'deathtime', 'dob']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. Crear variables útiles
df['age'] = df['age'].clip(upper=90)                    # tope de edad
df['los_days'] = (df['dischtime'] - df['admittime']).dt.days
df['los_days'] = df['los_days'].fillna(0).clip(lower=0)

# Variable objetivo clara
df['mortality'] = df['hospital_expire_flag'].astype(int)

# 3. Variables categóricas importantes
df['gender'] = df['gender'].map({'M': 0, 'F': 1})
df['admission_type'] = df['admission_type'].astype('category').cat.codes
df['first_careunit'] = df['first_careunit'].astype('category').cat.codes

# 4. Manejo de valores faltantes
# Rellenar numéricos con mediana
num_cols = ['age', 'los_days', 'num_diagnoses']
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Rellenar categóricos con el valor más frecuente
cat_cols = ['gender', 'admission_type', 'first_careunit']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

# 5. Seleccionar columnas finales para el modelo
features = ['age', 'gender', 'admission_type', 'los_days', 
            'num_diagnoses', 'first_careunit']

X = df[features].copy()
y = df['mortality'].copy()

print("\n=== DATASET FINAL LISTO PARA MODELOS ===")
print("Features (X):", X.shape)
print("Target (y):", y.shape)
print("\nColumnas usadas:", features)
print("\nValores faltantes restantes:\n", X.isnull().sum())

# Guardar versión final limpia
X.to_csv('X_mimic.csv', index=False)
y.to_csv('y_mimic.csv', index=False)

print("\nArchivos guardados: X_mimic.csv y y_mimic.csv")
print("Listo para usar con Regresión Logística y Redes Neuronales")

Dimensiones originales: (129, 27)

=== DATASET FINAL LISTO PARA MODELOS ===
Features (X): (129, 6)
Target (y): (129,)

Columnas usadas: ['age', 'gender', 'admission_type', 'los_days', 'num_diagnoses', 'first_careunit']

Valores faltantes restantes:
 age               0
gender            0
admission_type    0
los_days          0
num_diagnoses     0
first_careunit    0
dtype: int64

Archivos guardados: X_mimic.csv y y_mimic.csv
Listo para usar con Regresión Logística y Redes Neuronales
